# Vígil.ia — Melhorar a qualidade das bounding boxes

Implementa o plano `melhorar_boxes.md` na ordem certa:

1. **Etapa 1 — ajustes de inferência (grátis):** varredura imgsz × iou com saída visual → escolher a melhor config SEM retreinar
2. **Etapa 2 — auditoria de anotações:** box frouxa na inferência quase sempre é box frouxa no label; amostras visuais + estatísticas
3. **Etapa 3 — retreino focado em localização** (só se 1 e 2 não bastarem): YOLO11x com `box=10`; RT-DETR com patch no `loss_gain` (o argumento `box=` NÃO afeta o RT-DETR — a loss dele usa ganhos internos `{class:1, bbox:5, giou:2}`, verificado no fonte 8.4.80)
4. **Etapa 4 — filtros de pós-processamento** (roda em paralelo): filtro de área + ROI

⛔ **Não pule pra Etapa 3 antes de esgotar 1 e 2** — é a regra do plano.

Pré-requisitos no Drive: `soja_yolo11x_v3.pt` (campeão), `teste_soja.mp4`.
Opcionais: `soja_rtdetr_ft_v3.pt`, `soja_yolo11x_base.pt`, `soja_rtdetr_base.pt`, fotos reais (p/ Etapas 2–3).

## 0. Setup e caminhos

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')

YOLOX_PT = '/content/drive/MyDrive/soja_yolo11x_v3.pt'
assert os.path.exists(YOLOX_PT), 'soja_yolo11x_v3.pt não encontrado no Drive!'

RTDETR_PT = '/content/drive/MyDrive/soja_rtdetr_ft_v3.pt'
if not os.path.exists(RTDETR_PT):
    RTDETR_PT = None
    print('(aviso) RT-DETR não está no Drive — varredura só com o YOLO11x')

VIDEO_TESTE = '/content/drive/MyDrive/teste_soja.mp4'

# Se você AINDA estiver na sessão do tira-teima, salve o peso do estágio base
# do 11x (necessário pra Etapa 3 ser um experimento isolado):
for src in ['/content/runs/detect/runs_cmp/yolo11x_base/weights/best.pt']:
    if os.path.exists(src) and not os.path.exists('/content/drive/MyDrive/soja_yolo11x_base.pt'):
        !cp {src} /content/drive/MyDrive/soja_yolo11x_base.pt
        print('backup do estágio base feito: soja_yolo11x_base.pt')

REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']

# ETAPA 1 — Varredura de inferência (grátis, rodar primeiro)

Extrai frames de teste do vídeo e roda cada modelo com imgsz ∈ {640, 960, 1280}
× iou ∈ {0.45, 0.5, 0.6}. As imagens anotadas vão pra pastas separadas + um zip
no Drive pra você comparar visualmente.

In [ ]:
import cv2

# frames de teste: espalhados ao longo do vídeo (troque por uma pasta sua se preferir)
FRAMES_DIR = '/content/frames_teste'
N_FRAMES = 10
os.makedirs(FRAMES_DIR, exist_ok=True)
if not glob.glob(f'{FRAMES_DIR}/*.jpg'):
    cap = cv2.VideoCapture(VIDEO_TESTE)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for k, idx in enumerate(range(total // (N_FRAMES + 1), total, total // (N_FRAMES + 1))):
        if k >= N_FRAMES:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            cv2.imwrite(f'{FRAMES_DIR}/frame_{k:02d}.jpg', frame)
    cap.release()
print('frames de teste:', len(glob.glob(f'{FRAMES_DIR}/*.jpg')))

In [ ]:
import shutil
import torchvision
from ultralytics import YOLO, RTDETR
from ultralytics.models.rtdetr.predict import RTDETRPredictor

# Patch NMS do RT-DETR com iou AJUSTÁVEL (o predictor dele não roda NMS nativo)
RTDETR_NMS_IOU = 0.6
if not hasattr(RTDETRPredictor, '_pp_orig'):
    RTDETRPredictor._pp_orig = RTDETRPredictor.postprocess

def _pp_nms(self, preds, img, orig_imgs):
    results = RTDETRPredictor._pp_orig(self, preds, img, orig_imgs)
    for r in results:
        if len(r.boxes) > 1:
            keep = torchvision.ops.nms(r.boxes.xyxy, r.boxes.conf, iou_threshold=RTDETR_NMS_IOU)
            r.update(boxes=r.boxes.data[keep])
    return results

RTDETRPredictor.postprocess = _pp_nms

MODELS = {'yolo11x': YOLO(YOLOX_PT)}
if RTDETR_PT:
    MODELS['rtdetr'] = RTDETR(RTDETR_PT)

SWEEP = '/content/sweep_boxes'
shutil.rmtree(SWEEP, ignore_errors=True)

for tag, model in MODELS.items():
    for sz in (640, 960, 1280):
        for iou in (0.45, 0.5, 0.6):
            name = f'{tag}_imgsz{sz}_iou{iou}'
            if tag == 'rtdetr':
                globals()['RTDETR_NMS_IOU'] = iou   # NMS do patch
                model.predict(source=FRAMES_DIR, imgsz=sz, conf=0.35,
                              save=True, project=SWEEP, name=name,
                              exist_ok=True, verbose=False, device=0)
            else:
                model.predict(source=FRAMES_DIR, imgsz=sz, iou=iou, conf=0.35,
                              agnostic_nms=True, save=True, project=SWEEP,
                              name=name, exist_ok=True, verbose=False, device=0)
            print('ok:', name)

shutil.make_archive('/content/sweep_boxes_resultados', 'zip', SWEEP)
!cp /content/sweep_boxes_resultados.zip /content/drive/MyDrive/
print('\nZIP com todas as combinações no Drive: sweep_boxes_resultados.zip')

In [ ]:
# Grade comparativa de UM frame (o mesmo) em todas as configs do YOLO11x
import matplotlib.pyplot as plt

FRAME_REF = 'frame_04.jpg'   # troque pra inspecionar outro
cfgs = [(sz, iou) for sz in (640, 960, 1280) for iou in (0.45, 0.5, 0.6)]
plt.figure(figsize=(16, 14))
for i, (sz, iou) in enumerate(cfgs):
    p = f'{SWEEP}/yolo11x_imgsz{sz}_iou{iou}/{FRAME_REF}'
    if not os.path.exists(p):
        continue
    ax = plt.subplot(3, 3, i + 1)
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    ax.set_title(f'imgsz={sz}  iou={iou}', fontsize=10)
    ax.axis('off')
plt.suptitle('YOLO11x — mesma cena, 9 configs (procure a box mais justa)')
plt.tight_layout(); plt.show()

### Como ler a Etapa 1
- **imgsz maior** (960/1280) tende a apertar a box de grão pequeno — mas custa
  latência (~2,2× a 960, ~4× a 1280) e o modelo foi TREINADO em 640; se ficar
  estranho em 1280, é mismatch de escala, não bug.
- **iou menor** (0.45–0.5) mata box frouxa duplicada em grão colado; se grãos
  encostados começarem a SUMIR, o iou ficou agressivo demais.
- Anote a config vencedora — ela entra no pipeline (demo/track) e nos testes
  das etapas seguintes. **Se a box ficou justa o bastante aqui, PARE — não
  precisa retreinar.**

# ETAPA 2 — Auditoria das anotações (labels frouxos → boxes frouxas)
Requer o dataset v3 na sessão (`/content/soja_det_v3`). Se não estiver, rode as
células 1–4 do `melhoria_rtdetr_v3.ipynb` antes (constrói a partir do Drive).

In [ ]:
import numpy as np
import random

DS = '/content/soja_det_v3'
assert os.path.exists(f'{DS}/data.yaml'), (
    'Dataset v3 não está na sessão — rode melhoria_rtdetr_v3.ipynb células 1–4.')

def load_labels(split):
    rows = []  # (img_path, cls, cx, cy, w, h)
    for lp in glob.glob(f'{DS}/labels/{split}/*.txt'):
        ip = lp.replace('/labels/', '/images/').replace('.txt', '.jpg')
        for ln in open(lp):
            c, cx, cy, w, h = ln.split()
            rows.append((ip, int(c), float(cx), float(cy), float(w), float(h)))
    return rows

def audit(split):
    rows = load_labels(split)
    if not rows:
        return
    areas = np.array([w * h for *_, w, h in rows])
    touch = sum(1 for _, _, cx, cy, w, h in rows
                if cx - w/2 < 0.004 or cy - h/2 < 0.004 or cx + w/2 > 0.996 or cy + h/2 > 0.996)
    print(f'\n=== {split}: {len(rows)} caixas ===')
    print(f'  área (fração do frame): mediana={np.median(areas):.3f}  '
          f'p5={np.percentile(areas,5):.4f}  p95={np.percentile(areas,95):.3f}')
    print(f'  caixas GIGANTES (>30% do frame): {int((areas > 0.30).sum())} '
          f'({100*(areas > 0.30).mean():.1f}%)  <- suspeitas de label frouxo/errado')
    print(f'  caixas minúsculas (<0.2% do frame): {int((areas < 0.002).sum())}')
    print(f'  caixas tocando a borda: {touch} ({100*touch/len(rows):.1f}%)')
    per = {}
    for _, c, *_ , w, h in [(r[0], r[1], r[2], r[3], r[4], r[5]) for r in rows]:
        per.setdefault(c, []).append(w * h)
    for c in sorted(per):
        print(f'  {NAMES[c]:14s} n={len(per[c]):5d}  área mediana={np.median(per[c]):.3f}')
    plt.figure(figsize=(6, 3))
    plt.hist(areas, bins=60)
    plt.title(f'{split} — distribuição da área da caixa (fração do frame)')
    plt.tight_layout(); plt.show()

import matplotlib.pyplot as plt
for sp in ('train', 'val'):
    audit(sp)

In [ ]:
# Inspeção visual: amostras aleatórias com o ground-truth desenhado.
# Rode várias vezes (amostra nova a cada execução). Separa fotos reais de sintéticas.
def show_gt(prefix, title, n=6):
    paths = [p for p in glob.glob(f'{DS}/images/train/*.jpg')
             if os.path.basename(p).startswith(prefix)]
    random.shuffle(paths)
    plt.figure(figsize=(13, 8))
    for i, p in enumerate(paths[:n]):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        for ln in open(p.replace('/images/', '/labels/').replace('.jpg', '.txt')):
            c, cx, cy, ww, hh = ln.split()
            cx, cy, ww, hh = float(cx), float(cy), float(ww), float(hh)
            x1, y1 = int((cx - ww/2) * w), int((cy - hh/2) * h)
            cv2.rectangle(img, (x1, y1), (int((cx + ww/2) * w), int((cy + hh/2) * h)), (0, 255, 0), 2)
            cv2.putText(img, NAMES[int(c)], (x1, max(14, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)
        ax = plt.subplot(2, 3, i + 1); ax.imshow(img); ax.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_gt('train_', 'FOTOS REAIS — o label abraça o grão?')
show_gt('synth_', 'CENAS SINTÉTICAS — caixas justas nos grãos colados?')

### Como ler a Etapa 2
- Labels visivelmente frouxos/deslocados nas FOTOS REAIS → o problema está no
  pseudo-rótulo (padding do `sat_box` é ~4% + 2px; dá pra reduzir e reconstruir
  o v3) — me avise que ajusto o builder.
- Muitas caixas >30% do frame → label errado (o bug da "caixa gigante" que já
  tivemos) — me mande o print.
- Labels OK mas inferência frouxa mesmo na melhor config da Etapa 1 → aí sim a
  Etapa 3 se justifica.

# ETAPA 3 — Retreino focado em localização (SÓ depois das etapas 1–2)

Repete o estágio de fine-tune v3 mudando APENAS o peso da localização — mesmo
ponto de partida (estágio base), mesmos dados/épocas → efeito isolado.

- **YOLO11x:** `box=10.0` (padrão 7.5); geométrico já ativo; `close_mosaic=10`.
- **RT-DETR:** o argumento `box=` é IGNORADO pela loss dele — o patch abaixo
  dobra os ganhos internos de localização (`bbox` 5→10, `giou` 2→4).

In [ ]:
# --- YOLO11x com box=10 (a partir do estágio base, p/ isolar o efeito) ---
V3_YAML = '/content/soja_det_v3/data.yaml'
assert os.path.exists(V3_YAML), 'Dataset v3 não está na sessão (ver Etapa 2).'

YOLOX_BASE = '/content/runs/detect/runs_cmp/yolo11x_base/weights/best.pt'
if not os.path.exists(YOLOX_BASE):
    YOLOX_BASE = '/content/drive/MyDrive/soja_yolo11x_base.pt'
if not os.path.exists(YOLOX_BASE):
    YOLOX_BASE = YOLOX_PT
    print('⚠️ estágio base do 11x não encontrado — partindo do campeão v3.')
    print('   (comparável, mas menos isolado; lembre do caso ft_v3_disc)')

YOLO(YOLOX_BASE).train(
    data=V3_YAML, epochs=60, imgsz=640, batch=16, device=0, seed=42,
    optimizer='AdamW', lr0=0.001, patience=20,
    box=10.0,            # <- ÚNICA mudança de receita (padrão 7.5)
    close_mosaic=10,
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_box', name='yolo11x_v3_box10', exist_ok=True,
)
!cp /content/runs/detect/runs_box/yolo11x_v3_box10/weights/best.pt /content/drive/MyDrive/soja_yolo11x_v3_box10.pt
print('backup ok: soja_yolo11x_v3_box10.pt')

In [ ]:
# --- RT-DETR com loss de localização dobrada (patch no loss_gain) ---
# Verificado no fonte: RTDETRDetectionLoss usa loss_gain interno
# {class:1, bbox:5, giou:2}; o argumento box= do train() não chega nele.
from ultralytics.models.utils.loss import RTDETRDetectionLoss

if not hasattr(RTDETRDetectionLoss, '_init_orig'):
    RTDETRDetectionLoss._init_orig = RTDETRDetectionLoss.__init__

def _init_boxgain(self, *a, **kw):
    RTDETRDetectionLoss._init_orig(self, *a, **kw)
    self.loss_gain = {**self.loss_gain, 'bbox': 10, 'giou': 4}  # 5→10, 2→4
    print('loss_gain do RT-DETR ajustado p/ localização:', self.loss_gain)

RTDETRDetectionLoss.__init__ = _init_boxgain

RTDETR_BASE = '/content/drive/MyDrive/soja_rtdetr_base.pt'
assert os.path.exists(RTDETR_BASE), 'soja_rtdetr_base.pt não está no Drive.'

RTDETR(RTDETR_BASE).train(
    data=V3_YAML, epochs=40, imgsz=640, batch=12, device=0, seed=42,
    optimizer='AdamW', lr0=5e-5, patience=15, close_mosaic=8,
    amp=False, warmup_epochs=0.0,   # receita anti-colapso obrigatória
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_box', name='rtdetr_v3_boxgain', exist_ok=True,
)
!cp /content/runs/detect/runs_box/rtdetr_v3_boxgain/weights/best.pt /content/drive/MyDrive/soja_rtdetr_v3_boxgain.pt
print('backup ok: soja_rtdetr_v3_boxgain.pt')

# desfaz o patch (não vazar pro resto da sessão)
RTDETRDetectionLoss.__init__ = RTDETRDetectionLoss._init_orig

In [ ]:
# A/B da Etapa 3: mAP50-95 (a métrica de box justa) antes vs depois
print('mAP50-95 = quão MILIMETRICAMENTE justa a caixa é (a métrica desta task)\n')
pares = [('yolo11x v3 (atual)', YOLO(YOLOX_PT)),
         ('yolo11x box=10', YOLO('/content/runs/detect/runs_box/yolo11x_v3_box10/weights/best.pt'))]
if RTDETR_PT and os.path.exists('/content/runs/detect/runs_box/rtdetr_v3_boxgain/weights/best.pt'):
    pares += [('rtdetr v3 (atual)', RTDETR(RTDETR_PT)),
              ('rtdetr boxgain', RTDETR('/content/runs/detect/runs_box/rtdetr_v3_boxgain/weights/best.pt'))]
for tag, m in pares:
    r = m.val(data=V3_YAML, split='val', imgsz=640, device=0, verbose=False)
    print(f'{tag:22s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')

# ETAPA 4 — Filtros de pós-processamento (independente do treino)

- **Filtro de área:** mata a caixa gigante da borda da mesa/transição de fundo.
- **ROI:** infere só na região da bandeja (ignora fundo, caderno, sujeira).

Funções prontas pro pipeline + demonstração no vídeo com veredito travado.

In [ ]:
# ---- funções de filtro (copiar pro pipeline final) ----
MAX_AREA_FRAC = 0.15   # caixa maior que 15% do frame = lixo (ajuste ao teu cenário)
ROI = (0.05, 0.05, 0.95, 0.95)  # (x1,y1,x2,y2) fração do frame; None = desligado

def filtra_area(xyxy, frame_w, frame_h, max_frac=MAX_AREA_FRAC):
    """True = caixa passa (não é gigante)."""
    x1, y1, x2, y2 = xyxy
    return (x2 - x1) * (y2 - y1) <= max_frac * frame_w * frame_h

def recorta_roi(frame, roi=ROI):
    """Recorta a ROI antes de inferir. Devolve (recorte, offset_x, offset_y)."""
    if roi is None:
        return frame, 0, 0
    h, w = frame.shape[:2]
    x1, y1 = int(roi[0] * w), int(roi[1] * h)
    x2, y2 = int(roi[2] * w), int(roi[3] * h)
    return frame[y1:y2, x1:x2], x1, y1

# visualiza a ROI no 1º frame pra calibrar
cap = cv2.VideoCapture(VIDEO_TESTE)
ok, fr = cap.read()
cap.release()
h, w = fr.shape[:2]
vis = fr.copy()
cv2.rectangle(vis, (int(ROI[0]*w), int(ROI[1]*h)), (int(ROI[2]*w), int(ROI[3]*h)), (0, 255, 255), 3)
plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('ROI atual (amarelo) — ajuste a tupla ROI até cobrir só a bandeja')
plt.axis('off'); plt.show()

In [ ]:
# Demonstração: vídeo com veredito travado + ROI + filtro de área
from collections import defaultdict, Counter

BEST_IMGSZ = 640   # <- coloque a config vencedora da Etapa 1
BEST_IOU = 0.5
LOCK_MIN_FRAMES, LOCK_RATIO = 8, 0.60
COLORS = {'intact': (80, 200, 80), 'immature': (60, 200, 200),
          'broken': (200, 100, 160), 'skin-damaged': (60, 160, 255),
          'spotted': (80, 80, 230)}

model = MODELS['yolo11x']
votes, seen, locked = defaultdict(Counter), Counter(), {}
cap = cv2.VideoCapture(VIDEO_TESTE)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
writer = None
OUT = '/content/video_boxes_filtrado.mp4'

while True:
    ok, frame = cap.read()
    if not ok:
        break
    if writer is None:
        H, W = frame.shape[:2]
        writer = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    crop, ox, oy = recorta_roi(frame)
    r = model.track(crop, imgsz=BEST_IMGSZ, iou=BEST_IOU, conf=0.35,
                    agnostic_nms=True, tracker='bytetrack.yaml',
                    persist=True, verbose=False)[0]
    if r.boxes.id is not None:
        for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                                    r.boxes.id.int().tolist(),
                                    r.boxes.cls.int().tolist(),
                                    r.boxes.conf.tolist()):
            x1, y1, x2, y2 = xyxy
            x1, y1, x2, y2 = int(x1 + ox), int(y1 + oy), int(x2 + ox), int(y2 + oy)
            if not filtra_area((x1, y1, x2, y2), W, H):
                continue  # caixa gigante -> lixo
            if tid not in locked:
                votes[tid][NAMES[c]] += cf
                seen[tid] += 1
                top, n = votes[tid].most_common(1)[0]
                if seen[tid] >= LOCK_MIN_FRAMES and n >= LOCK_RATIO * sum(votes[tid].values()):
                    locked[tid] = top
            cls = locked.get(tid)
            color = COLORS[cls] if cls else (160, 160, 160)
            label = f'#{tid} {cls}' if cls else f'#{tid} analisando…'
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, max(18, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    cv2.rectangle(frame, (int(ROI[0]*W), int(ROI[1]*H)), (int(ROI[2]*W), int(ROI[3]*H)),
                  (0, 255, 255), 1)
    writer.write(frame)
cap.release()
writer.release()

for tid, cnt in votes.items():
    locked.setdefault(tid, cnt.most_common(1)[0][0])
intact = sum(1 for c in locked.values() if c == 'intact')
print(f'grãos: {len(locked)} | intactos: {intact}')
!cp {OUT} /content/drive/MyDrive/
print('vídeo no Drive: video_boxes_filtrado.mp4')

## Checklist final

1. Etapa 1: escolheu a config vencedora (imgsz/iou)? Anote-a — ela vai pro
   `demo_servidor_colab.ipynb` e pro pipeline de track.
2. Etapa 2: labels justos? Se não, o conserto é no BUILDER (me avise), não no treino.
3. Etapa 3: só rode se 1–2 não bastarem. Compare pelo **mAP50-95** e pelo vídeo.
4. Etapa 4: calibre `ROI` e `MAX_AREA_FRAC` no teu cenário e me diga os valores
   — eu integro no servidor de demo e no auto-treino v4.